# "THE PRICE IS RIGHT" - Dự án Capstone

## Tóm tắt quy trình của notebook

Notebook này xây dựng một hệ thống ước lượng giá sản phẩm từ mô tả văn bản của sản phẩm, dựa trên dữ liệu đã được crawl từ Amazon. Quy trình đi theo các bước: tải dữ liệu, chuyển văn bản thành vector, huấn luyện mạng nơ-ron, và cuối cùng so sánh với các mô hình LLM frontier.

## Ý nghĩa chính của notebook

Notebook này giải quyết bài toán: “Nếu tôi biết mô tả của một sản phẩm, liệu có thể ước lượng được nó đáng bao nhiêu tiền không?” Dữ liệu đầu vào là `summary` (mô tả ngắn về sản phẩm), còn nhãn là `price` (giá thực tế). Quá trình này cho thấy cách một mô hình có thể học từ dữ liệu văn bản để ước tính giá.

Mô hình được đánh giá theo khả năng dự đoán trên tập test. Kết quả cuối cùng không chỉ cho thấy độ chính xác của mô hình mà còn giúp so sánh giữa mô hình huấn luyện từ đầu, baseline thủ công và các mô hình LLM không cần huấn luyện lại.

## Mục tiêu cuối cùng

Mục tiêu của notebook là giúp người học hiểu được:
- dữ liệu văn bản có thể được biểu diễn dưới dạng số học
- mạng nơ-ron có thể học mối quan hệ giữa mô tả và giá
- LLM cũng có thể được dùng như một công cụ ước lượng giá bằng prompt
- hiệu quả của mỗi phương pháp phải được đánh giá bằng số liệu thực tế

> Notebook này thực hiện chuỗi từ dữ liệu -> biểu diễn -> học máy -> đánh giá -> so sánh mô hình, đây là dạng quy trình chuẩn của một dự án AI thực tế.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# Trước khi xem xét mạng nơ-ron nhân tạo

## Có một kiểu mạng nơ-ron khác mà ta có thể cân nhắc

Trước khi bắt đầu xây dựng mạng nơ-ron, ta cần hiểu một điều quan trọng: không phải mọi bài toán đều cần mô hình học máy truyền thống theo cách cổ điển. Với dữ liệu văn bản, ta có thể biến mô tả thành vector và cho mô hình học trực tiếp từ dữ liệu để tìm mối liên hệ với giá sản phẩm.

Cell này đặt nền móng cho việc hiểu rằng dữ liệu text không chỉ là chuỗi ký tự mà còn có thể được biểu diễn dưới dạng số và được dùng làm đầu vào cho mô hình học sâu.

> Cell này là bước chuyển tiếp từ “xử lý dữ liệu bằng thủ công” sang “để mô hình tự học các đặc trưng từ dữ liệu”.

In [ ]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


In [ ]:
evaluate(human_pricer, test, size=100)

# Bây giờ là một mạng nơ-ron cơ bản

Trong phần còn lại của notebook, ta sẽ đi sâu vào cách mạng nơ-ron hoạt động và cách huấn luyện chúng. Đây không phải là kiến thức đầy đủ về deep learning, mà là một bản xem lướt ngắn để hiểu ý tưởng cốt lõi trước khi đi vào các mô hình lớn hơn.

Ta sẽ tự xây dựng một mạng nơ-ron bằng PyTorch từ đầu, chỉ với mục tiêu tạo cảm giác trực quan về quá trình học. Dữ liệu văn bản sẽ được chuyển thành vector, sau đó được đưa vào các lớp nơ-ron để dự đoán giá.

> Cell này là khởi đầu cho phần deep learning, giúp kết nối giữa dữ liệu và mô hình học qua các lớp ẩn.

In [ ]:
# Chuẩn bị dữ liệu đầu vào và nhãn mục tiêu

# Tạo mảng giá trị mục tiêu y từ tập huấn luyện
# Mỗi item trong train có thuộc tính price và summary
# Giá trị price là nhãn cần dự đoán, còn summary là mô tả văn bản

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

# Đối với mỗi sản phẩm, ta hình thành một cặp: mô tả sản phẩm -> giá thật.
# Đây là dạng dữ liệu mà mô hình học máy cần để học mối quan hệ giữa text và số tiền.

In [ ]:
# Dùng HashingVectorizer để chuyển văn bản thành vector số
# Với binary=True, mỗi từ chỉ được đánh dấu là có mặt hoặc không, thay vì đếm số lần xuất hiện.

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

# Kết quả là mỗi mô tả sản phẩm được biểu diễn dưới dạng vector có chiều cố định 5000.
# Đây là dạng mà mạng nơ-ron có thể hiểu được, vì mô hình chỉ làm việc với con số, không với văn bản thuần.

In [ ]:
# Khai báo kiến trúc mạng nơ-ron bằng PyTorch

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        # Mỗi lớp tuyến tính biến đổi không gian đặc trưng của đầu vào
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Propagation xuôi: dữ liệu đi qua từng lớp và áp dụng ReLU
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

# Mạng này chỉ là một ví dụ đơn giản để minh họa cách deep learning xử lý dữ liệu văn bản.
# Quan trọng nhất là hiểu rằng dữ liệu đi qua nhiều lớp, mỗi lớp trích xuất các đặc trưng khác nhau và cuối cùng cho ra giá dự đoán.

In [ ]:
# Chuyển dữ liệu thành tensor của PyTorch
# X_train_tensor: ma trận đặc trưng của tập huấn luyện
# y_train_tensor: giá mục tiêu, thêm chiều thứ nhất vì đầu ra của mô hình là 1 giá trị

X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Chia tập dữ liệu thành train và validation
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Tạo DataLoader để đưa dữ liệu theo từng batch vào mô hình
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Khởi tạo mô hình với số đầu vào bằng số chiều của vector
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

# Bây giờ mô hình đã sẵn sàng để bắt đầu quá trình học từ dữ liệu đã được mã hóa.

In [ ]:
# Đếm số tham số có thể học của mô hình
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

# Đây là tổng số trọng số mà mô hình sẽ cập nhật trong quá trình huấn luyện.
# Nếu số lượng lớn, mô hình có nhiều khả năng học nhưng cũng cần nhiều dữ liệu và thời gian hơn.

In [ ]:
# Định nghĩa hàm mất mát và optimizer
# MSELoss đo khoảng cách giữa giá dự đoán và giá thật
# Adam giúp cập nhật trọng số theo hướng giảm lỗi hiệu quả hơn

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Huấn luyện 2 epoch để xem mô hình cải thiện như thế nào
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # 4 bước chính của quá trình huấn luyện:
        # 1. Forward pass: đưa dữ liệu qua mạng để tạo output
        # 2. Tính loss: so sánh output với nhãn thật
        # 3. Backward pass: tính gradient
        # 4. Optimizer step: cập nhật trọng số
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

# Mục tiêu là giảm loss trên tập validation, nghĩa là mô hình đang học được quy luật dữ liệu và dự đoán tốt hơn.

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        # Chuyển mô tả sản phẩm thành vector bằng cùng vectorizer đã dùng ở trên
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

# Hàm này dùng để dự đoán giá cho một sản phẩm mới dựa trên mô tả của nó.
# Vì giá không thể âm nên nếu mô hình dự đoán âm, ta clamp về 0.

In [ ]:
evaluate(neural_network, test)

# Bây giờ là các mô hình frontier

Hãy xem cách các mô hình frontier hoạt động khi không cần huấn luyện lại gì cả. Chúng chỉ suy luận dựa trên tri thức đã được học trước đó trong quá trình pretraining.

Ngày mai, ta sẽ thực hiện fine-tuning cho một mô hình frontier để cải thiện độ phù hợp với bài toán cụ thể.

> Đây là điểm khác biệt lớn giữa deep learning truyền thống và LLM: một mô hình truyền thống cần học từ dữ liệu mới, còn LLM có thể suy luận trực tiếp từ kiến thức nền có sẵn.

In [ ]:
def messages_for(item):
    # Mô hình được yêu cầu ước tính giá của sản phẩm dựa trên mô tả của nó.
    # Chúng ta yêu cầu trả về con số, không giải thích dài dòng.
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

# Cell này chuẩn hóa prompt để gửi cho mọi mô hình LLM trong notebook.

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [ ]:
# Hàm dự đoán cho gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

# Đây là cách dùng LLM như một bộ ước lượng giá mà không cần huấn luyện lại model.
# Mô hình sẽ dựa trên prompt và tri thức nền có sẵn để suy đoán giá sản phẩm.

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test)

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

# Claude Opus 4.5 là một mô hình mạnh, được sử dụng để so sánh trực tiếp với các model khác trong bài toán dự đoán giá.

In [ ]:
evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

# Gemini được thử với mức reasoning_effort thấp để kiểm tra hiệu suất nhanh và đơn giản.
# Mục tiêu là thấy khi tập trung vào tốc độ, mô hình có giữ được chất lượng dự đoán hay không.

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

# Model này là phiên bản nhẹ hơn, phù hợp cho việc kiểm tra tốc độ và chi phí cùng lúc.
# Khi so sánh với mô hình mạnh hơn, ta có thể thấy rõ trade-off giữa hiệu suất và tính hiệu quả.

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:
def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

# Grok được thử thêm để mở rộng so sánh giữa nhiều nền tảng LLM khác nhau trên cùng bài toán.

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# Hàm dự đoán cho gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content

# Đây là thử nghiệm với model mạnh nhất trong danh sách, dùng mức suy luận cao để xem khả năng hiểu sâu nhất của LLM.

In [ ]:
evaluate(gpt_5__1, test)